In [ ]:
# Step 1: Setup and Installation
!pip install -q peft transformers datasets
from transformers import AutoModelForSequenceClassification, AutoTokenizer, default_data_collator, get_linear_schedule_with_warmup
from peft import get_peft_config, get_peft_model, PromptTuningInit, PromptTuningConfig, TaskType
import torch
from torch.nn.functional import pad
from datasets import load_dataset , Dataset
import pandas as pd
from torch.utils.data import DataLoader
from tqdm import tqdm

# Step 2: Define Configuration
device = "cuda"
model_name_or_path = 'bert-base-multilingual-uncased'
tokenizer_name_or_path = 'bert-base-multilingual-uncased'
peft_config = PromptTuningConfig(
    task_type=TaskType.SEQ_CLS,  # Correct task type for sequence classification
    prompt_tuning_init=PromptTuningInit.TEXT,
    num_virtual_tokens=8,
    prompt_tuning_init_text="What is the sentiment of this text?",  # Changed to a more specific prompt
    tokenizer_name_or_path=tokenizer_name_or_path,
)
dataset_name = "twitter_complaints"
checkpoint_name = f"{dataset_name}_{model_name_or_path}_{peft_config.peft_type}_{peft_config.task_type}_v1.pt".replace("/", "_")
text_column = "Tweet text"
label_column = "text_label"
max_length = 128  # Increased max_length to accommodate longer texts
lr = 3e-2
num_epochs = 20
batch_size = 8

# Step 3: Load Dataset

# dataset = pd.read_csv()

# dataset.iloc[0]

column_names = [label_column,"blank" ,text_column]
# Load the dataset without headers and assign the column names
dataset = pd.read_csv("/content/drive/MyDrive/societal_2l_train_hi.csv", header=None, names=column_names).head(500) #examples in dataset
dataset.drop("blank", axis=1, inplace=True)
dataset[label_column] = dataset[label_column].map({0: 0, 4: 1})  # Map labels to 0 and 1 for binary classification
print(dataset.head())

from sklearn.model_selection import train_test_split

# Split the preprocessed dataset into training and testing sets (80% train, 20% test)
train_dataset, test_dataset = train_test_split(dataset, test_size=0.2, random_state=42)

# Now you have train_dataset and test_dataset for training and testing respectively
print(train_dataset)
dataset_train = Dataset.from_pandas(train_dataset)
dataset_test = Dataset.from_pandas(test_dataset)

# Step 4: Tokenizer Initialization
tokenizer = AutoTokenizer.from_pretrained(model_name_or_path)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

# Step 5: Preprocess Function
def preprocess_function(examples):
    model_inputs = tokenizer(examples[text_column], truncation=True, max_length=max_length, padding="max_length")
    model_inputs["labels"] = examples[label_column]
    return model_inputs

# Step 6: Apply Preprocessing
processed_datasets_train = dataset_train.map(
    preprocess_function,
    batched=True,
    num_proc=1,
    remove_columns=dataset_train.column_names,
    load_from_cache_file=False,
    desc="Running tokenizer on dataset",
)
processed_datasets_test = dataset_test.map(
    preprocess_function,
    batched=True,
    num_proc=1,
    remove_columns=dataset_test.column_names,
    load_from_cache_file=False,
    desc="Running tokenizer on dataset",
)

# Step 7: DataLoaders
train_dataloader = DataLoader(
    processed_datasets_train, shuffle=True, collate_fn=default_data_collator, batch_size=batch_size, pin_memory=True
)
test_dataloader = DataLoader(
    processed_datasets_test, shuffle=False, collate_fn=default_data_collator, batch_size=batch_size, pin_memory=True
)

# Step 8: Model Initialization
model = AutoModelForSequenceClassification.from_pretrained(model_name_or_path, num_labels=2)  # Binary classification
model = get_peft_model(model, peft_config)
print(model.print_trainable_parameters())

# Step 9: Optimizer and Scheduler
optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
lr_scheduler = get_linear_schedule_with_warmup(optimizer=optimizer, num_warmup_steps=0, num_training_steps=(len(train_dataloader) * num_epochs))

# Step 10: Training Loop
model = model.to(device)
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for step, batch in enumerate(tqdm(train_dataloader)):
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        total_loss += loss.detach().float()
        loss.backward()
        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()

    model.eval()
    eval_loss = 0
    eval_preds = []
    for step, batch in enumerate(tqdm(test_dataloader)):
        batch = {k: v.to(device) for k, v in batch.items()}
        with torch.no_grad():
            outputs = model(**batch)
        loss = outputs.loss
        eval_loss += loss.detach().float()
        eval_preds.extend(torch.argmax(outputs.logits, -1).detach().cpu().numpy())

    eval_epoch_loss = eval_loss / len(test_dataloader)
    train_epoch_loss = total_loss / len(train_dataloader)
    print(f"Epoch {epoch}: train_loss={train_epoch_loss}, eval_loss={eval_epoch_loss}")


   text_label                                         Tweet text
0           0  @sardanarohit @sudhirchaudhary @arnabgoswamias...
1           1  @narendramodi प्रिय पीएम, युवाओं को हम #Cashle...
2           0  @Narendramodi सर पठानकोट निगम AMRIT CITY SCHEO...
3           1  #jobs #jobsearch ##parliament GST बिल पास करता...
4           1  @Upsubramanya ppl praise manmohan 4 1991 सुधार...
     text_label                                         Tweet text
249           1  #GST को अच्छी तरह से समझें और इसे अपने निर्वाच...
433           1  @YADAVAKHILESH आप असली SAMAJWADI हैं। ऐसे व्यक...
19            0  यदि पाक के लोग आतंकवाद के खिलाफ खड़े होने के ल...
322           0  Pathankot के दागी SP SALWINDER ने @Timesofindi...
332           0  @Pmoindia, @finminindia @narendramodi केंद्र स...
..          ...                                                ...
106           0  RT @kumar_ke5hav: #surgicalStrike और एक कमजोर ...
270           0  @airtelindia @ideacellular @vodafonein @aircel...
348    

Running tokenizer on dataset:   0%|          | 0/400 [00:00<?, ? examples/s]

Running tokenizer on dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-multilingual-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


trainable params: 7,682 || all params: 167,365,636 || trainable%: 0.0046
None


...

Epoch 17: train_loss=0.5783738493919373, eval_loss=0.6806333661079407


100%|██████████| 13/13 [00:00<00:00, 15.07it/s]


Epoch 18: train_loss=0.4980086386203766, eval_loss=0.6929308176040649


100%|██████████| 13/13 [00:00<00:00, 14.76it/s]

Epoch 19: train_loss=0.4877986013889313, eval_loss=0.6611429452896118


In [ ]:
inputs = tokenizer(
    f'{text_column} : {"मुझे इस फिल्म से बहुत निराशा हुई। यह बिल्कुल भी अच्छी नहीं थी।"} Label : ',
    return_tensors="pt",
)

In [ ]:
model.to(device)

with torch.no_grad():
    inputs = {k: v.to(device) for k, v in inputs.items()}
    outputs = model(**inputs)
    logits = outputs.logits
    predictions = torch.argmax(logits, dim=1)
    print(predictions.detach().cpu().numpy())

[0]
